# Olist Brazilian E-Commerce Analysis

Imports and Engine Set-up

In [ ]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import URL, create_engine
from dotenv import load_dotenv

load_dotenv()

url_object = URL.create(
    "postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=5433,
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url_object)

FIGURES_PATH = Path("../figures/")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", palette="crest")
plt.rcParams["figure.figsize"] = (10, 6)

### Question 1: Which Order categoriy drive the most revenue?

In [ ]:
Q1_REVENUE_BY_CATEGORY = """
    SELECT
        SUM(price) AS total_revenue,
        product_category_name_english AS category
    FROM
        order_items
    INNER JOIN products
        ON order_items.product_id = products.product_id
    INNER JOIN category_name_translation
        ON products.product_category_name = category_name_translation.product_category_name
    GROUP BY product_category_name_english
    ORDER BY total_revenue DESC
    LIMIT 10;
"""

with engine.connect() as conn:
    df_revenue = pd.read_sql(Q1_REVENUE_BY_CATEGORY, con=conn)

df_revenue.head()

In [ ]:
df_revenue_sorted = df_revenue.sort_values("total_revenue", ascending=True)
df_revenue_sorted["total_revenue"] = df_revenue_sorted.total_revenue / 1000000

plt.barh(df_revenue_sorted.category, df_revenue_sorted.total_revenue)

plt.title("Olist E-Commerce Revenue by Categories")
plt.ylabel("Order Categories")
plt.xlabel("Total Revenue (R$)")

plt.savefig(FIGURES_PATH / "01_orders_revenue_by_categories.png", bbox_inches="tight")
plt.show()

**What it shows:** Health & beauty leads category revenue at BRL 1.26M, followed closely by watches/gifts (BRL 1.20M) and bed/bath/table (BRL 1.04M), while all remaining categories fall below BRL 1M.

**What it means:** Olist's customers primarily purchase personal care and home lifestyle products, suggesting the platform's strongest market is everyday consumer goods rather than electronics or industrial categories.

**What to do about it:** Olist should investigate whether underperforming categories suffer from a supply (seller) or demand (buyer) gap, and explore whether top-performing categories can scale further through additional seller recruitment, targeted promotions, or geographic expansion into underserved states.

In [ ]:
Q2_SELLER_DELAYS = """
    SELECT
        sellers.seller_id,
        sellers.seller_city,
        sellers.seller_state,
        ROUND(AVG(order_delivered_customer_date::DATE - order_estimated_delivery_date::DATE), 1) AS avg_seller_day_delay,
        COUNT (*) AS total_late_orders
    FROM order_items
    INNER JOIN orders
        ON order_items.order_id = orders.order_id
    INNER JOIN sellers
        ON order_items.seller_id = sellers.seller_id
    WHERE
        order_status = 'delivered'
    AND
        order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
    GROUP BY
        sellers.seller_id,
        sellers.seller_city,
        sellers.seller_state
    HAVING COUNT(*) >= 10
    ORDER BY avg_seller_day_delay DESC;
    """
with engine.connect() as conn:
    df_delays_days = pd.read_sql(Q2_SELLER_DELAYS, con=conn)


df_delays_days.head()

In [ ]:
avg_delay_days_sorted = df_delays_days.sort_values(
    "avg_seller_day_delay", ascending=True
)
avg_delay_days_sorted["seller_identity"] = (
    avg_delay_days_sorted.seller_city + ", " + avg_delay_days_sorted.seller_state
)
top_10_delays = avg_delay_days_sorted.tail(10)

plt.barh(width=top_10_delays.avg_seller_day_delay, y=top_10_delays.seller_identity)

plt.title("Olist Top 10 Average Seller Delay Days")
plt.xlabel("Average Delay Days")
plt.ylabel("Seller City and State")

plt.savefig(FIGURES_PATH / "02_seller_delivery_delays.png", bbox_inches="tight")

**What it shows:** The chart above shows the 10 worst-performing sellers. Extending the analysis to the top 20 sellers exceeding the delay threshold shows the same pattern: delays are spread across 5 states, with São Paulo (SP) accounting for 11 of the top 20 worst offenders, followed by Minas Gerais (4), Paraná (2), Santa Catarina (2), and Rio de Janeiro (1); the single worst seller, in Belo Horizonte (MG), averages 39.3 days late across 15 orders, followed closely by sellers in Indaiatuba, SP (32.5 days, 15 orders) and São José dos Campos, SP (32.2 days, 16 orders).

**What it means:** Delivery delays seem to be caused by individual seller performance rather than a single regional logistics issue because the worst offenders are dispersed throughout several states rather than concentrated in one area, and every seller on this list has at least ten delayed orders, ruling out one-time errors.

**What to do about it:** To ascertain whether the delays are resolvable seller-side problems or an indication that these sellers should be flagged or removed from the platform, Olist should look into the particular operations of these repeat offenders, their shipping strategies, order volume in relation to capacity, and courier partnerships.

In [ ]:
Q3_AVG_REVIEW_DELIVERY_TIME = """
    SELECT
        order_reviews.review_score,
        ROUND(AVG(order_delivered_customer_date::DATE - order_approved_at::DATE), 1) AS avg_delivery_days
    FROM
        orders
    INNER JOIN
        order_reviews
            ON orders.order_id = order_reviews.order_id
    WHERE 
        orders.order_status = 'delivered'
    GROUP BY
        order_reviews.review_score
    ORDER BY avg_delivery_days DESC;
"""
with engine.connect() as conn:
    df_avg_review_dlivery_time = pd.read_sql(Q3_AVG_REVIEW_DELIVERY_TIME, con=conn)


df_avg_review_dlivery_time.head()

In [ ]:
df_sorted_by_score = df_avg_review_dlivery_time.sort_values("review_score")

plt.bar(df_sorted_by_score.review_score, df_sorted_by_score.avg_delivery_days)

plt.title("Average Delivery Time by Review Score")
plt.xlabel("Review Score Rating")
plt.ylabel("Average Delivery Time (Days)")

plt.savefig(FIGURES_PATH / "03_review_score_vs_delivery_time.png", bbox_inches="tight")

**What it shows:** The chart demonstrates a clear, monotonic relationship between delivery speed and customer satisfaction, where 1-star reviews average a lengthy 20.7 days to arrive, while 5-star ratings drop steadily down to 10.1 days, effectively cutting the customer wait time in half from the worst rating to the best.

**What it means:** Shipping speed acts as a massive driver of customer sentiment, but this highlights a correlation rather than a proven causation. While slow delivery heavily frustrates buyers, other underlying issues such as damaged items or poor product quality frequently compound the frustration and drive down low ratings.

**What to do about it:** Operations teams should establish strict SLA benchmarks aimed closer to the 10-day window seen in 5-star reviews, while customer success and product teams should audit review text to separate logistics complaints from product-level defects.

In [ ]:
Q4_STATE_VALUE = """
    SELECT
        customers.customer_state,
        SUM(order_payment.payment_value) AS total_state_value
    FROM
        orders
    INNER JOIN
        customers
            ON orders.customer_id = customers.customer_id
    INNER JOIN
        order_payment
            ON orders.order_id = order_payment.order_id
    GROUP BY
        customers.customer_state
    ORDER BY total_state_value DESC;
    """

with engine.connect() as conn:
    df_state_revenue = pd.read_sql(Q4_STATE_VALUE, con=conn)

df_state_revenue.head()

In [ ]:
df_state_revenue["revenue_in_millions"] = (
    df_state_revenue["total_state_value"] / 1000000
)
df_sorted = df_state_revenue.sort_values("revenue_in_millions", ascending=True).tail(10)

plt.barh(df_sorted.customer_state, df_sorted.revenue_in_millions)

plt.xlabel("Revenue in Millions")
plt.ylabel("State")
plt.title("Customers Revenue By State")

plt.savefig(FIGURES_PATH / "04_customer_revenue_by_state.png", bbox_inches="tight")

**What it shows:** E-commerce revenue is heavily skewed geographically, with São Paulo (SP) leading the platform at roughly 6.0 million dollars more than double Rio de Janeiro (RJ) at 2.1 million dollars followed by a steep drop-off across the remaining regions, proving that the top three states carry the vast majority of overall sales.

**What it means:** Olist’s high-value customer base is deeply concentrated in the country’s core economic hubs rather than distributed evenly nationwide, meaning regional economic power and dense urban logistics heavily dictate platform revenue.

What to do about it: Marketing and logistics strategies should double down on these core regions (SP, RJ, and adjacent high-yield states) for high-return customer acquisition campaigns, while operations teams investigate whether lower-revenue states suffer from poor carrier infrastructure or simply lack market penetration.